# California DEM — Exploratory Data Analysis

Static **Copernicus DEM GLO-30** terrain features clipped to the **California state polygon** (not the rectangular bbox), plus how to fuse them with your ERA5 time series.

**DEM layers:** elevation, slope, aspect, hillshade, TRI, TPI  
**ERA5 join key:** sample DEM once per 0.25° cell → join to every `(time, lat, lon)` ERA5 row

## 0. Setup

In [ ]:
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
import seaborn as sns
from matplotlib.colors import LightSource
from rasterio.plot import show

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 120

# Resolve project root whether notebook is opened from eda/ or project root
CWD = Path.cwd()
ROOT = CWD if (CWD / "eda" / "outputs").exists() else CWD.parent
OUT = ROOT / "eda" / "outputs"
NUM = OUT / "numerical"
CLIP = OUT / "clipped_ca"
FIG = OUT / "figures"
FIG.mkdir(parents=True, exist_ok=True)

LAYERS = ["elevation", "slope", "aspect", "hillshade", "tri", "tpi"]
NODATA = -32768.0

ERA5_VARS = [
    "2m_temperature",
    "2m_dewpoint_temperature",
    "surface_pressure",
    "10m_u_component_of_wind",
    "10m_v_component_of_wind",
    "instantaneous_10m_wind_gust",
    "total_precipitation",
    "volumetric_soil_water_layer_1",
    "volumetric_soil_water_layer_2",
    "high_vegetation_cover",
    "low_vegetation_cover",
    "leaf_area_index_high_vegetation",
    "leaf_area_index_low_vegetation",
    "boundary_layer_height",
]

print("ROOT:", ROOT)
print("Numerical:", NUM.exists(), "| Clipped CA:", CLIP.exists())

In [ ]:
summary = pd.read_csv(NUM / "california_terrain_summary.csv")
sample = pd.read_parquet(NUM / "california_terrain_sample.parquet")
era5_dem = pd.read_parquet(NUM / "era5_grid_dem_features.parquet")
corr = pd.read_csv(NUM / "terrain_correlation.csv", index_col=0)
boundary = gpd.read_file(ROOT / "eda" / "boundaries" / "california.geojson")

print("Summary rows:", len(summary))
print("Pixel sample:", sample.shape)
print("ERA5 grid cells:", era5_dem.shape)
display(summary.round(2))

## 1. Summary statistics (California polygon only)

These stats are from terrain rasters **masked to the official CA state outline**, not the full study bbox.

In [ ]:
plot_cols = ["mean", "median", "std", "min", "max"]
fig, ax = plt.subplots(figsize=(10, 4))
summary.set_index("layer")[plot_cols].plot(kind="bar", ax=ax)
ax.set_title("California DEM — key statistics by layer")
ax.set_ylabel("Value")
ax.tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.savefig(FIG / "nb_summary_bars.png", dpi=140, bbox_inches="tight")
plt.show()

summary[["layer", "count", "mean", "median", "std", "p05", "p95"]].round(2)

## 2. Value distributions

Using a random sample of California pixels (up to 500k) for interactive plotting.

In [ ]:
plot_df = sample.sample(n=min(80_000, len(sample)), random_state=42)

fig, axes = plt.subplots(2, 3, figsize=(13, 7))
for ax, col in zip(axes.ravel(), LAYERS):
    ax.hist(plot_df[col].dropna(), bins=70, color="#1f6f6a", alpha=0.9, edgecolor="none")
    ax.axvline(plot_df[col].median(), color="#c44e52", ls="--", lw=1.2, label="median")
    ax.set_title(col)
    ax.set_xlabel(col)
    ax.set_ylabel("count")
    ax.legend(fontsize=8)
fig.suptitle("Terrain value distributions — California state mask", fontsize=13)
fig.tight_layout()
plt.savefig(FIG / "nb_histograms.png", dpi=140, bbox_inches="tight")
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(13, 7))
for ax, col in zip(axes.ravel(), LAYERS):
    sns.kdeplot(plot_df[col].dropna(), ax=ax, fill=True, color="#3d7ea6", alpha=0.55)
    ax.set_title(col)
fig.suptitle("Kernel density estimates (California DEM sample)", fontsize=13)
fig.tight_layout()
plt.savefig(FIG / "nb_kde.png", dpi=140, bbox_inches="tight")
plt.show()

## 3. Correlations between terrain features

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 6))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdBu_r", center=0, ax=ax, square=True)
ax.set_title("Pearson correlation — California DEM features")
plt.tight_layout()
plt.savefig(FIG / "nb_correlation.png", dpi=140, bbox_inches="tight")
plt.show()

print("Strongest pairs (|r|):")
pairs = (
    corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
    .stack()
    .abs()
    .sort_values(ascending=False)
)
display(pairs.head(8).round(3))

In [ ]:
# Pair relationships most useful for wildfire / ERA5 fusion
pair_cols = ["elevation", "slope", "tri", "tpi"]
g = sns.pairplot(
    plot_df[pair_cols].sample(n=min(12_000, len(plot_df)), random_state=0),
    corner=True,
    plot_kws={"s": 4, "alpha": 0.25, "color": "#2f5d50"},
    diag_kws={"color": "#2f5d50"},
)
g.fig.suptitle("Pairwise terrain relationships (sample)", y=1.02)
plt.savefig(FIG / "nb_pairplot.png", dpi=120, bbox_inches="tight")
plt.show()

## 4. Spatial maps (California-clipped rasters)

Maps use the CA-masked GeoTIFFs (~300 m EDA resolution).

In [ ]:
def read_masked(path: Path):
    with rasterio.open(path) as src:
        data = src.read(1).astype(np.float32)
        data = np.where(data == NODATA, np.nan, data)
        extent = [src.bounds.left, src.bounds.right, src.bounds.bottom, src.bounds.top]
    return data, extent

cmaps = {
    "elevation": "terrain",
    "slope": "YlOrRd",
    "aspect": "twilight",
    "hillshade": "gray",
    "tri": "magma",
    "tpi": "coolwarm",
}

fig, axes = plt.subplots(2, 3, figsize=(14, 10))
for ax, layer in zip(axes.ravel(), LAYERS):
    data, extent = read_masked(CLIP / f"{layer}_ca.tif")
    im = ax.imshow(data, extent=extent, origin="upper", cmap=cmaps[layer], aspect="auto")
    boundary.boundary.plot(ax=ax, color="black", linewidth=0.6)
    ax.set_title(layer)
    ax.set_xlabel("lon")
    ax.set_ylabel("lat")
    plt.colorbar(im, ax=ax, fraction=0.035, pad=0.02)
fig.suptitle("California DEM terrain layers (state polygon mask)", fontsize=14)
fig.tight_layout()
plt.savefig(FIG / "nb_spatial_maps.png", dpi=140, bbox_inches="tight")
plt.show()

In [ ]:
# Elevation × slope relationship spatially (bivariate view via scatter of grid cells)
elev, extent = read_masked(CLIP / "elevation_ca.tif")
slope, _ = read_masked(CLIP / "slope_ca.tif")

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
im0 = axes[0].imshow(elev, extent=extent, origin="upper", cmap="terrain", aspect="auto")
boundary.boundary.plot(ax=axes[0], color="k", lw=0.6)
axes[0].set_title("Elevation (m)")
plt.colorbar(im0, ax=axes[0], fraction=0.04)

im1 = axes[1].imshow(slope, extent=extent, origin="upper", cmap="YlOrRd", aspect="auto")
boundary.boundary.plot(ax=axes[1], color="k", lw=0.6)
axes[1].set_title("Slope (degrees)")
plt.colorbar(im1, ax=axes[1], fraction=0.04)

fig.suptitle("Elevation vs slope — California")
fig.tight_layout()
plt.savefig(FIG / "nb_elev_slope_maps.png", dpi=140, bbox_inches="tight")
plt.show()

## 5. DEM sampled onto ERA5 0.25° grid

This is the **static lookup table** you join to every ERA5 timestep.

In [ ]:
display(era5_dem.head())
print(f"Cells: {len(era5_dem)} | Valid elevation: {era5_dem['elevation'].notna().sum()}")
era5_dem[LAYERS].describe().round(2)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 10))
for ax, layer in zip(axes.ravel(), LAYERS):
    sc = ax.scatter(
        era5_dem["longitude"],
        era5_dem["latitude"],
        c=era5_dem[layer],
        s=22,
        cmap=cmaps[layer],
        alpha=0.95,
    )
    boundary.boundary.plot(ax=ax, color="black", linewidth=0.7)
    ax.set_title(f"{layer} @ ERA5 cells")
    ax.set_xlabel("lon")
    ax.set_ylabel("lat")
    plt.colorbar(sc, ax=ax, fraction=0.04)
fig.suptitle("Static DEM features on ERA5 0.25° grid (California)", fontsize=14)
fig.tight_layout()
plt.savefig(FIG / "nb_era5_grid_maps.png", dpi=140, bbox_inches="tight")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
hb = ax.hexbin(
    era5_dem["elevation"],
    era5_dem["slope"],
    gridsize=35,
    cmap="viridis",
    mincnt=1,
)
ax.set_xlabel("Elevation (m)")
ax.set_ylabel("Slope (deg)")
ax.set_title("ERA5 cells: elevation vs slope")
plt.colorbar(hb, ax=ax, label="cell count")
plt.tight_layout()
plt.savefig(FIG / "nb_era5_elev_slope_hex.png", dpi=140, bbox_inches="tight")
plt.show()

## 6. How to use DEM with your ERA5 time series

DEM is **static**. Your ERA5 fields are **spatiotemporal**. Fusion pattern:

```text
ERA5(t, lat, lon, weather...)  ⟕  DEM(lat, lon, elevation, slope, ...)
         └──────── join on lat/lon (or cell_id) ────────┘
```

In [ ]:
fusion_notes = pd.DataFrame(
    [
        {"ERA5 variable": "2m_temperature / 2m_dewpoint_temperature", "DEM help": "Elevation (lapse rate), slope/aspect (local heating)"},
        {"ERA5 variable": "surface_pressure", "DEM help": "Strongly tied to elevation (hydrostatic)"},
        {"ERA5 variable": "10m wind U/V + gust", "DEM help": "Slope, TRI, TPI → exposure, channeling, ridges"},
        {"ERA5 variable": "total_precipitation", "DEM help": "Orographic lift from elevation + slope orientation"},
        {"ERA5 variable": "soil water L1/L2", "DEM help": "Slope/TPI → runoff vs retention (valleys wetter)"},
        {"ERA5 variable": "vegetation cover / LAI", "DEM help": "Elevation/aspect zones as prior for fuel/veg structure"},
        {"ERA5 variable": "boundary_layer_height", "DEM help": "Elevation + roughness (TRI) modulate BLH"},
    ]
)
display(fusion_notes)

print("Your ERA5 feature list:")
for v in ERA5_VARS:
    print(" -", v)

In [ ]:
# Example join pattern (replace era5_df with your real ERA5 dataframe)
demo_era5 = pd.DataFrame(
    {
        "time": pd.to_datetime(["2023-07-01"] * 5),
        "latitude": era5_dem["latitude"].iloc[:5].values,
        "longitude": era5_dem["longitude"].iloc[:5].values,
        "2m_temperature": np.random.uniform(285, 310, 5),
        "total_precipitation": np.random.uniform(0, 5, 5),
    }
)

dem_lookup = era5_dem.set_index(["latitude", "longitude"])[LAYERS]
joined = demo_era5.join(dem_lookup, on=["latitude", "longitude"])

print("Example joined ERA5 + DEM row:")
display(joined.head())

# Useful derived features for wildfire models
joined["wind_speed_proxy_note"] = "hypot(u10, v10) from ERA5"
era5_dem["sin_aspect"] = np.sin(np.deg2rad(era5_dem["aspect"]))
era5_dem["cos_aspect"] = np.cos(np.deg2rad(era5_dem["aspect"]))
era5_dem["orographic_index"] = era5_dem["elevation"] * era5_dem["slope"]

display(era5_dem[["elevation", "slope", "sin_aspect", "cos_aspect", "orographic_index"]].describe().round(2))

## 7. Takeaways

1. **Use California-masked terrain**, not the full bbox (ocean / NV / AZ edges removed).
2. **Primary ML inputs from DEM:** elevation, slope, aspect (use sin/cos), TRI, TPI.
3. **Join strategy:** `era5_grid_dem_features.parquet` → left-join onto every ERA5 `(time, lat, lon)`.
4. **Vegetation ERA5 fields** are slow-changing; DEM still adds topographic context they lack.
5. **Next step:** merge real ERA5 2021–2025 with this DEM lookup and FIRMS labels for training.

### Key files
- `eda/outputs/numerical/california_terrain_summary.csv`
- `eda/outputs/numerical/california_terrain_sample.parquet`
- `eda/outputs/numerical/era5_grid_dem_features.parquet`
- `eda/outputs/reports/dem_era5_fusion.md`